In [14]:
# ============ CONFIG ============
import os

CONFIG = {
    "annotations_root": "landmarks",   # recursively searched for *.xml (CVAT exports)
    "images_root": "../../selected_photos",                  # recursively searched for the actual jpg/png files
    "work_dir": "work",

    "iris_labels": {"left": "left_iris_mask", "right": "right_iris_mask"},
    # corner/white-point labels used ONLY to build a sensible crop box per eye (not trained on)
    "corner_labels": {
        "left":  ["left_eye_inner_corner", "left_eye_outer_corner", "left_white_point"],
        "right": ["right_eye_inner_corner", "right_eye_outer_corner", "right_white_point"],
    },

    "crop_margin": 1.8,     # crop box = corner-bbox expanded by this factor (>1 gives context around the eye)
    "crop_size": 256,       # resize each eye crop to this size for the U-Net

    "batch_size": 8,
    "epochs_frozen": 15,
    "epochs_finetune": 40,
    "lr_frozen": 1e-3,
    "lr_finetune": 1e-4,
    "val_fraction": 0.2,
    "seed": 42,
    "encoder": "resnet18",
    "device": "cuda",
}

for sub in ("masks", "checkpoints", "predictions", "crops"):
    os.makedirs(os.path.join(CONFIG["work_dir"], sub), exist_ok=True)
print(CONFIG)


{'annotations_root': 'landmarks', 'images_root': '../../selected_photos', 'work_dir': 'work', 'iris_labels': {'left': 'left_iris_mask', 'right': 'right_iris_mask'}, 'corner_labels': {'left': ['left_eye_inner_corner', 'left_eye_outer_corner', 'left_white_point'], 'right': ['right_eye_inner_corner', 'right_eye_outer_corner', 'right_white_point']}, 'crop_margin': 1.8, 'crop_size': 256, 'batch_size': 8, 'epochs_frozen': 15, 'epochs_finetune': 40, 'lr_frozen': 0.001, 'lr_finetune': 0.0001, 'val_fraction': 0.2, 'seed': 42, 'encoder': 'resnet18', 'device': 'cuda'}


In [3]:
import numpy as np, random, glob
import matplotlib.pyplot as plt
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter



In [15]:
def find_xml_files(root):
    return sorted(glob.glob(os.path.join(root, "**", "*.xml"), recursive=True))

def index_images(root, exts=(".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG")):
    index = {}
    for ext in exts:
        for p in glob.glob(os.path.join(root, "**", f"*{ext}"), recursive=True):
            index[os.path.basename(p)] = p
    return index

xml_files = find_xml_files(CONFIG["annotations_root"])
image_index = index_images(CONFIG["images_root"])
print(f"Found {len(xml_files)} annotation XML files")
print(f"Indexed {len(image_index)} image files")


Found 15 annotation XML files
Indexed 420 image files


In [28]:
def parse_points_str(points_str):
    pts = []
    for pair in points_str.strip().split(';'):
        x_str, y_str = pair.split(',')
        pts.append((float(x_str), float(y_str)))
    return np.array(pts, dtype=np.float32)

def fit_circle(points):
    x, y = points[:, 0], points[:, 1]
    A = np.column_stack([x, y, np.ones_like(x)])
    b = x**2 + y**2
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy = sol[0] / 2.0, sol[1] / 2.0
    r = np.sqrt(sol[2] + cx**2 + cy**2)
    return float(cx), float(cy), float(r)

def robust_parse_xml(xml_path):
    """Try the standard parser first; fall back to lxml's recovering parser
    (tolerates stray control chars, bad entities, minor truncation) if that fails.
    Returns an ElementTree root, or None if the file is unsalvageable."""
    try:
        return ET.parse(xml_path).getroot()
    except ET.ParseError as e:
        try:
            from lxml import etree as LET
            parser = LET.XMLParser(recover=True)
            tree = LET.parse(xml_path, parser=parser)
            print(f"[warn] {xml_path}: recovered from parse error ({e}) using lxml recover mode")
            return tree.getroot()
        except Exception as e2:
            print(f"[SKIPPED - could not parse] {xml_path}: {e} / lxml fallback also failed: {e2}")
            return None


def normalize_label(label):
    """Case/spacing/dash-insensitive normalization so 'Left_Iris_Mask', 'left iris mask',
    'left-iris-mask' etc. all match the same canonical key."""
    if label is None:
        return ""
    return label.strip().lower().replace("-", "_").replace(" ", "_")


def ellipse_to_points(ellipse_tag, n=16):
    """Convert a CVAT <ellipse> shape (cx, cy, rx, ry, rotation) into n boundary points,
    so it can flow through the same fit_circle / fillPoly code as polygon-based iris labels."""
    cx = float(ellipse_tag.get("cx"))
    cy = float(ellipse_tag.get("cy"))
    rx = float(ellipse_tag.get("rx"))
    ry = float(ellipse_tag.get("ry"))
    rotation_deg = float(ellipse_tag.get("rotation", 0.0))
    theta = np.deg2rad(rotation_deg)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
    ex = rx * np.cos(angles)
    ey = ry * np.sin(angles)
    # apply rotation, then translate to center
    xs = cx + ex * np.cos(theta) - ey * np.sin(theta)
    ys = cy + ex * np.sin(theta) + ey * np.cos(theta)
    return np.column_stack([xs, ys]).astype(np.float32)


# def parse_all_xml(xml_files, iris_labels, corner_labels):
    """Returns dict: filename -> {
        'width', 'height',
        'left':  {'iris_pts':..., 'cx','cy','r', 'corner_pts':...}  (if present)
        'right': {...} (if present)
    }"""
    iris_labels_norm = {side: normalize_label(lbl) for side, lbl in iris_labels.items()}
    corner_labels_norm = {side: [normalize_label(l) for l in labels] for side, labels in corner_labels.items()}

    records = {}
    unmatched_labels = Counter()

    for xml_path in xml_files:
        root = robust_parse_xml(xml_path)
        if root is None:
            continue
        for image_tag in root.findall("image"):
            filename = image_tag.get("name")
            width = float(image_tag.get("width"))
            height = float(image_tag.get("height"))

            rec = records.setdefault(filename, {"width": width, "height": height})

            # gather all point-like AND ellipse shapes by NORMALIZED label
            shapes_by_norm_label = {}
            for tag_name in ("points", "polygon", "polyline", "ellipse"):
                for shape_tag in image_tag.findall(tag_name):
                    norm = normalize_label(shape_tag.get("label"))
                    shapes_by_norm_label.setdefault(norm, []).append(shape_tag)
                    unmatched_labels[(shape_tag.tag, shape_tag.get("label"))] += 1

            for side, norm_iris_label in iris_labels_norm.items():
                if norm_iris_label not in shapes_by_norm_label:
                    continue
                shape_tag = shapes_by_norm_label[norm_iris_label][0]
                if shape_tag.tag == "ellipse":
                    iris_pts = ellipse_to_points(shape_tag)
                else:
                    iris_pts = parse_points_str(shape_tag.get("points"))
                if len(iris_pts) < 5:
                    continue
                unmatched_labels.pop((shape_tag.tag, shape_tag.get("label")), None)
                cx, cy, r = fit_circle(iris_pts)

                corner_pts = []
                corner_dict = {}
                for norm_label, orig_label in zip(corner_labels_norm[side], corner_labels[side]):
                    if norm_label in shapes_by_norm_label:
                        ctag = shapes_by_norm_label[norm_label][0]
                        unmatched_labels.pop((ctag.tag, ctag.get("label")), None)
                        if ctag.tag == "ellipse":
                            pt = [float(ctag.get("cx")), float(ctag.get("cy"))]
                        else:
                            pt = parse_points_str(ctag.get("points"))[0].tolist()
                        corner_pts.append(pt)
                        corner_dict[orig_label] = tuple(pt)  # e.g. corner_dict["left_eye_inner_corner"] = (x, y)
                corner_pts = np.array(corner_pts, dtype=np.float32) if corner_pts else None

                rec[side] = {
                    "iris_pts": iris_pts, "cx": cx, "cy": cy, "r": r,
                    "corner_pts": corner_pts,
                    "corner_dict": corner_dict,  # NEW: named access, e.g. corner_dict["left_eye_outer_corner"]
                }

    if unmatched_labels:
        print("Shapes found that did NOT match any configured iris/corner label "
              "(check for naming variants you may want to add to CONFIG):")
        for (tag, label), count in unmatched_labels.most_common(20):
            print(f"  {count:5d}  <{tag}> label={label!r}")

    return records

def parse_all_xml(xml_files, iris_labels, corner_labels):
    iris_labels_norm = {side: normalize_label(lbl) for side, lbl in iris_labels.items()}
    corner_labels_norm = {side: [normalize_label(l) for l in labels] for side, labels in corner_labels.items()}

    records = {}
    unmatched_labels = Counter()

    for xml_path in xml_files:
        root = robust_parse_xml(xml_path)
        if root is None:
            continue

        for image_tag in root.findall("image"):
            filename = image_tag.get("name")
            width = float(image_tag.get("width"))
            height = float(image_tag.get("height"))

            rec = records.setdefault(filename, {"width": width, "height": height})

            shapes_by_norm_label = {}

            for tag_name in ("points", "polygon", "polyline", "ellipse"):
                for shape_tag in image_tag.findall(tag_name):
                    norm = normalize_label(shape_tag.get("label"))
                    shapes_by_norm_label.setdefault(norm, []).append(shape_tag)
                    unmatched_labels[(shape_tag.tag, shape_tag.get("label"))] += 1

            # اول برای هر سمت ساختار پایه بساز
            for side in ("left", "right"):
                rec.setdefault(side, {
                    "iris_pts": None,
                    "cx": None,
                    "cy": None,
                    "r": None,
                    "corner_pts": None,
                    "corner_dict": {},
                })

            # استخراج cornerها، مستقل از iris
            for side in ("left", "right"):
                corner_pts = []
                corner_dict = {}

                for norm_label, orig_label in zip(corner_labels_norm[side], corner_labels[side]):
                    if norm_label in shapes_by_norm_label:
                        ctag = shapes_by_norm_label[norm_label][0]
                        unmatched_labels.pop((ctag.tag, ctag.get("label")), None)

                        if ctag.tag == "ellipse":
                            pt = [float(ctag.get("cx")), float(ctag.get("cy"))]
                        else:
                            pt = parse_points_str(ctag.get("points"))[0].tolist()

                        corner_pts.append(pt)
                        corner_dict[orig_label] = tuple(pt)

                if corner_pts:
                    rec[side]["corner_pts"] = np.array(corner_pts, dtype=np.float32)
                    rec[side]["corner_dict"] = corner_dict

            # استخراج iris، اگر وجود داشت
            for side, norm_iris_label in iris_labels_norm.items():
                if norm_iris_label not in shapes_by_norm_label:
                    continue

                shape_tag = shapes_by_norm_label[norm_iris_label][0]

                if shape_tag.tag == "ellipse":
                    iris_pts = ellipse_to_points(shape_tag)
                else:
                    iris_pts = parse_points_str(shape_tag.get("points"))

                if len(iris_pts) < 5:
                    continue

                unmatched_labels.pop((shape_tag.tag, shape_tag.get("label")), None)

                cx, cy, r = fit_circle(iris_pts)

                rec[side]["iris_pts"] = iris_pts
                rec[side]["cx"] = cx
                rec[side]["cy"] = cy
                rec[side]["r"] = r

    if unmatched_labels:
        print("Shapes found that did NOT match any configured iris/corner label "
              "(check for naming variants you may want to add to CONFIG):")
        for (tag, label), count in unmatched_labels.most_common(20):
            print(f"  {count:5d}  <{tag}> label={label!r}")

    return records

records = parse_all_xml(xml_files, CONFIG["iris_labels"], CONFIG["corner_labels"])
n_left = sum(1 for r in records.values() if "left" in r)
n_right = sum(1 for r in records.values() if "right" in r)
print(f"Images with parsed annotations: {len(records)}  (left iris: {n_left}, right iris: {n_right})")


Images with parsed annotations: 419  (left iris: 419, right iris: 419)


## 12. Human labeling error (second annotator on 42 random test photos)

You have a second, independent set of labels for 42 randomly-chosen photos, in one XML file. Comparing these to the original labels on the *same* photos gives an estimate of **human labeling noise** — how much two people (or the same person on two passes) naturally disagree when placing the same points. This matters because it sets a realistic floor: if the model's error is close to or below this human-vs-human disagreement, the model isn't meaningfully "wrong" anymore — it's within the noise of the labels themselves.


In [29]:
# ============ point this at your second-label file ============
CONFIG["retest_xml_path"] = "landmarks_test_human_error/annotations.xml"   # <-- update to your actual file

retest_records = parse_all_xml([CONFIG["retest_xml_path"]], CONFIG["iris_labels"], CONFIG["corner_labels"])
print(f"Parsed {len(retest_records)} images from the retest file")

common_filenames = [f for f in retest_records if f in records]
print(f"Filenames present in BOTH the original labels and the retest file: {len(common_filenames)}")
if len(common_filenames) < len(retest_records):
    missing = set(retest_records) - set(records)
    print(f"  ({len(missing)} retest filenames have no matching original label - check filename spelling if this is unexpected)")
retest_records


Parsed 42 images from the retest file
Filenames present in BOTH the original labels and the retest file: 41
  (1 retest filenames have no matching original label - check filename spelling if this is unexpected)


{'P010_IMG_4718.JPG': {'width': 3024.0,
  'height': 4032.0,
  'left': {'iris_pts': None,
   'cx': None,
   'cy': None,
   'r': None,
   'corner_pts': array([[1826.8 , 2104.9 ],
          [1701.13, 2086.3 ]], dtype=float32),
   'corner_dict': {'left_eye_outer_corner': (1826.800048828125,
     2104.89990234375),
    'left_white_point': (1701.1300048828125, 2086.300048828125)}},
  'right': {'iris_pts': None,
   'cx': None,
   'cy': None,
   'r': None,
   'corner_pts': array([[1338.75, 2147.5 ],
          [1080.12, 2138.36],
          [1201.98, 2111.04]], dtype=float32),
   'corner_dict': {'right_eye_inner_corner': (1338.75, 2147.5),
    'right_eye_outer_corner': (1080.1199951171875, 2138.360107421875),
    'right_white_point': (1201.97998046875, 2111.0400390625)}}},
 'P020_IMG_7974.JPG': {'width': 3024.0,
  'height': 4032.0,
  'left': {'iris_pts': None,
   'cx': None,
   'cy': None,
   'r': None,
   'corner_pts': array([[1700.22, 1958.72],
          [1987.1 , 1935.8 ],
          [1841.85,

In [33]:
def compare_two_label_sets(filenames, records_a, records_b, label_a="original", label_b="retest"):
    """Returns a dict of lists: per-side center distance, radius diff, and per-corner-label distance."""
    results = {
        "left_center_dist": [], "left_radius_diff": [],
        "right_center_dist": [], "right_radius_diff": [],
        "corner_dist_by_label": {},  # label -> list of distances
    }

    for filename in filenames:
        rec_a, rec_b = records_a[filename], records_b[filename]
        for side in ("left", "right"):
            if side not in rec_a or side not in rec_b:
                continue
            a, b = rec_a[side], rec_b[side]

            # center_dist = float(np.hypot(a["cx"] - b["cx"], a["cy"] - b["cy"]))
            # radius_diff = float(abs(a["r"] - b["r"]))
            # results[f"{side}_center_dist"].append(center_dist)
            # results[f"{side}_radius_diff"].append(radius_diff)

            # per-corner comparison, matched by label name (robust to missing points on either side)
            dict_a, dict_b = a.get("corner_dict", {}), b.get("corner_dict", {})
            for label in set(dict_a) & set(dict_b):
                pa, pb = dict_a[label], dict_b[label]
                dist = float(np.hypot(pa[0] - pb[0], pa[1] - pb[1]))
                results["corner_dist_by_label"].setdefault(label, []).append(dist)

    return results


human_error = compare_two_label_sets(common_filenames, records, retest_records)

def print_stats(name, values):
    if not values:
        print(f"{name}: no data")
        return
    values = np.array(values)
    print(f"{name:28s} n={len(values):3d}  mean={values.mean():6.2f}  median={np.median(values):6.2f}  "
          f"std={values.std():6.2f}  max={values.max():6.2f}")

# print("=== Iris center distance between the two annotators (full-image px) ===")
# print_stats("left eye", human_error["left_center_dist"])
# print_stats("right eye", human_error["right_center_dist"])

# print("\n=== Iris radius difference between the two annotators (px) ===")
# print_stats("left eye", human_error["left_radius_diff"])
# print_stats("right eye", human_error["right_radius_diff"])

print("\n=== Corner point distance between the two annotators (px), by label ===")
for label, dists in human_error["corner_dist_by_label"].items():
    print_stats(label, dists)





=== Corner point distance between the two annotators (px), by label ===
left_eye_outer_corner        n= 41  mean=  5.00  median=  4.73  std=  3.01  max= 13.20
left_white_point             n= 41  mean=  0.36  median=  0.30  std=  0.23  max=  1.22
right_white_point            n= 41  mean=  0.31  median=  0.27  std=  0.18  max=  0.92
right_eye_inner_corner       n= 41  mean=  2.45  median=  2.25  std=  1.20  max=  6.39
right_eye_outer_corner       n= 41  mean=  5.05  median=  4.50  std=  2.87  max= 15.84
left_eye_inner_corner        n= 38  mean=  3.33  median=  2.93  std=  2.44  max=  9.76
